# Phase 2: Stochastic Modelling & Derivatives Pricing  
## Notebook 02_10 — Finite Difference PDE Pricer

### Research question

How can the Black-Scholes-Merton PDE be solved numerically using finite differences, and how do stability, boundary conditions, grid refinement, and early exercise affect derivative prices?

This notebook links three Oxford-coded themes:

1. **PDEs** — the Black-Scholes-Merton pricing equation.
2. **Numerical methods** — explicit, implicit, and Crank-Nicolson finite differences.
3. **Derivatives pricing** — European and American option valuation.

The key idea:

> A closed-form option price is one solution of a PDE. A finite difference pricer turns that PDE into a flexible numerical engine.

## 1. Black-Scholes-Merton PDE

Under the risk-neutral measure:

$$
dS_t=(r-q)S_tdt+\sigma S_tdW_t
$$
The option value $V(t,S)$ satisfies:

$$
\begin{aligned}
&\frac{\partial V}{\partial t} \\
&\quad + \frac{1}{2}\sigma^2S^2 \frac{\partial^2V}{\partial S^2} \\
&\quad + (r-q)S \frac{\partial V}{\partial S} \\
&\quad - rV =0
\end{aligned}
$$
Using time-to-maturity:

$$
\tau=T-t
$$
the PDE becomes:

$$
\begin{aligned}
\frac{\partial V}{\partial \tau} &= \frac{1}{2}\sigma^2S^2V_{SS} \\
&\quad + (r-q)SV_S \\
&\quad - rV
\end{aligned}
$$
We solve forward in $\tau$, starting from the terminal payoff.

## 2. Boundary conditions

We solve on a finite grid:

$$
S\in[0,S_{\max}]
$$
For a European call:

$$
C(\tau,0)=0
$$
$$
C(\tau,S_{\max})\approx S_{\max}e^{-q\tau}-Ke^{-r\tau}
$$
For a European put:

$$
P(\tau,0)=Ke^{-r\tau}
$$
$$
P(\tau,S_{\max})\approx0
$$
For an American put:

$$
P_A(\tau,S)\geq(K-S)^+
$$
and at $S=0$:

$$
P_A(\tau,0)=K
$$

## 3. Finite difference operator

On a uniform grid:

$$
S_i=i\Delta S
$$
Central differences give:

$$
V_S\approx\frac{V_{i+1}-V_{i-1}}{2\Delta S}
$$
$$
V_{SS}\approx\frac{V_{i+1}-2V_i+V_{i-1}}{\Delta S^2}
$$
The PDE operator can be written:

$$
LV_i=a_iV_{i-1}+b_iV_i+c_iV_{i+1}
$$
where:

$$
a_i=\frac{1}{2}\sigma^2i^2-\frac{1}{2}(r-q)i
$$
$$
b_i=-\sigma^2i^2-r
$$
$$
c_i=\frac{1}{2}\sigma^2i^2+\frac{1}{2}(r-q)i
$$

## 4. Theta method

The theta method is:

$$
\begin{aligned}
(I-\theta\Delta\tau L)V^{n+1} &= (I+(1-\theta)\Delta\tau L)V^n
\end{aligned}
$$
| $\theta$ | Scheme |
|---:|---|
| 0 | Explicit |
| 1 | Implicit |
| 1/2 | Crank-Nicolson |

Crank-Nicolson is usually more accurate than implicit for smooth problems, while implicit is more damping and robust.

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from math import erf, exp, log, sqrt
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class FDConfig:
    s0: float = 100.0
    strike: float = 100.0
    maturity: float = 1.0
    r: float = 0.04
    q: float = 0.00
    sigma: float = 0.20
    smax_mult: float = 4.0
    m: int = 400
    n: int = 800
    option_type: str = "call"


cfg = FDConfig()
cfg

In [ ]:
def normal_cdf(x):
    x = np.asarray(x, dtype=float)
    return 0.5 * (1.0 + np.vectorize(erf)(x / np.sqrt(2.0)))


def normal_pdf(x):
    x = np.asarray(x, dtype=float)
    return np.exp(-0.5 * x**2) / np.sqrt(2.0 * np.pi)


def bsm_price(S, K, T, r, q, sigma, option_type):
    S = np.asarray(S, dtype=float)

    if T <= 0:
        return np.maximum(S - K, 0.0) if option_type == "call" else np.maximum(K - S, 0.0)

    S_safe = np.maximum(S, 1e-12)
    d1 = (np.log(S_safe / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        return S * np.exp(-q*T) * normal_cdf(d1) - K * np.exp(-r*T) * normal_cdf(d2)
    if option_type == "put":
        return K * np.exp(-r*T) * normal_cdf(-d2) - S * np.exp(-q*T) * normal_cdf(-d1)

    raise ValueError("option_type must be 'call' or 'put'")


def bsm_delta(S, K, T, r, q, sigma, option_type):
    d1 = (log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrt(T))
    if option_type == "call":
        return float(exp(-q*T) * normal_cdf(d1))
    if option_type == "put":
        return float(exp(-q*T) * (normal_cdf(d1) - 1.0))
    raise ValueError("option_type must be 'call' or 'put'")


def bsm_gamma(S, K, T, r, q, sigma):
    d1 = (log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrt(T))
    return float(exp(-q*T) * normal_pdf(d1) / (S * sigma * sqrt(T)))

In [ ]:
def make_grid(cfg):
    smax = cfg.smax_mult * max(cfg.s0, cfg.strike)
    S = np.linspace(0.0, smax, cfg.m + 1)
    dS = S[1] - S[0]
    dt = cfg.maturity / cfg.n
    tau = np.linspace(0.0, cfg.maturity, cfg.n + 1)
    return S, dS, dt, tau


def payoff(S, K, option_type):
    if option_type == "call":
        return np.maximum(S - K, 0.0)
    if option_type == "put":
        return np.maximum(K - S, 0.0)
    raise ValueError("option_type must be 'call' or 'put'")


def boundary(tau, smax, K, r, q, option_type, american=False):
    if option_type == "call":
        return 0.0, max(smax * exp(-q*tau) - K * exp(-r*tau), 0.0)
    if option_type == "put":
        return (K if american else K * exp(-r*tau)), 0.0
    raise ValueError("option_type must be 'call' or 'put'")


def operator_coeffs(cfg):
    i = np.arange(1, cfg.m)
    a = 0.5 * cfg.sigma**2 * i**2 - 0.5 * (cfg.r - cfg.q) * i
    b = -cfg.sigma**2 * i**2 - cfg.r
    c = 0.5 * cfg.sigma**2 * i**2 + 0.5 * (cfg.r - cfg.q) * i
    return a, b, c

In [ ]:
def thomas(lower, diag, upper, rhs):
    n = len(diag)
    cp = np.zeros(n - 1)
    dp = np.zeros(n)

    cp[0] = upper[0] / diag[0]
    dp[0] = rhs[0] / diag[0]

    for i in range(1, n - 1):
        denom = diag[i] - lower[i - 1] * cp[i - 1]
        cp[i] = upper[i] / denom
        dp[i] = (rhs[i] - lower[i - 1] * dp[i - 1]) / denom

    denom = diag[-1] - lower[-1] * cp[-1]
    dp[-1] = (rhs[-1] - lower[-1] * dp[-2]) / denom

    x = np.zeros(n)
    x[-1] = dp[-1]

    for i in range(n - 2, -1, -1):
        x[i] = dp[i] - cp[i] * x[i + 1]

    return x

In [ ]:
def explicit_fd(cfg):
    S, dS, dt, tau_grid = make_grid(cfg)
    smax = S[-1]
    V = payoff(S, cfg.strike, cfg.option_type)
    a, b, c = operator_coeffs(cfg)
    snapshots = {0.0: V.copy()}

    for n in range(cfg.n):
        tau_next = tau_grid[n + 1]
        old = V.copy()
        V[1:-1] = old[1:-1] + dt * (a * old[:-2] + b * old[1:-1] + c * old[2:])
        V[0], V[-1] = boundary(tau_next, smax, cfg.strike, cfg.r, cfg.q, cfg.option_type)

        if n + 1 in {cfg.n // 4, cfg.n // 2, 3 * cfg.n // 4, cfg.n}:
            snapshots[round(float(tau_next), 8)] = V.copy()

    return {
        "price": float(np.interp(cfg.s0, S, V)),
        "S": S,
        "V": V,
        "dS": dS,
        "dt": dt,
        "snapshots": snapshots,
        "stability_indicator": cfg.sigma**2 * cfg.m**2 * dt
    }

In [ ]:
def psor_lcp(lower, diag, upper, rhs, obstacle, omega=1.25, tol=1e-10, max_iter=20000):
    n = len(diag)
    x = np.maximum(rhs / diag, obstacle)

    for it in range(1, max_iter + 1):
        old = x.copy()

        for j in range(n):
            left = lower[j - 1] * x[j - 1] if j > 0 else 0.0
            right = upper[j] * old[j + 1] if j < n - 1 else 0.0
            continuation = (rhs[j] - left - right) / diag[j]
            relaxed = x[j] + omega * (continuation - x[j])
            x[j] = max(obstacle[j], relaxed)

        if np.max(np.abs(x - old)) < tol:
            return x, it

    return x, max_iter


def theta_fd(cfg, theta=0.5, american=False):
    if american and cfg.option_type != "put":
        raise ValueError("American solver here is implemented for puts only.")

    S, dS, dt, tau_grid = make_grid(cfg)
    smax = S[-1]
    V = payoff(S, cfg.strike, cfg.option_type)
    intrinsic = payoff(S, cfg.strike, cfg.option_type)
    snapshots = {0.0: V.copy()}

    a, b, c = operator_coeffs(cfg)

    lower = -theta * dt * a[1:]
    diag = 1.0 - theta * dt * b
    upper = -theta * dt * c[:-1]

    psor_iters = []

    for n in range(cfg.n):
        tau_old = tau_grid[n]
        tau_new = tau_grid[n + 1]

        lo_old, hi_old = boundary(tau_old, smax, cfg.strike, cfg.r, cfg.q, cfg.option_type, american)
        lo_new, hi_new = boundary(tau_new, smax, cfg.strike, cfg.r, cfg.q, cfg.option_type, american)

        old = V.copy()
        old[0], old[-1] = lo_old, hi_old

        rhs = old[1:-1] + (1 - theta) * dt * (a * old[:-2] + b * old[1:-1] + c * old[2:])
        rhs[0] += theta * dt * a[0] * lo_new
        rhs[-1] += theta * dt * c[-1] * hi_new

        if american:
            interior, iters = psor_lcp(lower, diag, upper, rhs, intrinsic[1:-1])
            psor_iters.append(iters)
        else:
            interior = thomas(lower, diag, upper, rhs)

        V[0], V[-1] = lo_new, hi_new
        V[1:-1] = interior

        if american:
            V = np.maximum(V, intrinsic)
            V[0], V[-1] = lo_new, hi_new

        if n + 1 in {cfg.n // 4, cfg.n // 2, 3 * cfg.n // 4, cfg.n}:
            snapshots[round(float(tau_new), 8)] = V.copy()

    return {
        "price": float(np.interp(cfg.s0, S, V)),
        "S": S,
        "V": V,
        "dS": dS,
        "dt": dt,
        "theta": theta,
        "american": american,
        "snapshots": snapshots,
        "psor_iterations": psor_iters
    }

## 5. European call validation

We compare explicit, implicit, and Crank-Nicolson prices against the analytical BSM reference.

In [ ]:
call_cfg = FDConfig(option_type="call", m=400, n=1600)

explicit_call = explicit_fd(call_cfg)
implicit_call = theta_fd(call_cfg, theta=1.0)
crank_call = theta_fd(call_cfg, theta=0.5)

bsm_call = float(bsm_price(
    call_cfg.s0, call_cfg.strike, call_cfg.maturity,
    call_cfg.r, call_cfg.q, call_cfg.sigma, "call"
))

call_comparison = pd.DataFrame([
    {
        "scheme": "explicit",
        "price": explicit_call["price"],
        "bsm_reference": bsm_call,
        "error": explicit_call["price"] - bsm_call,
        "abs_error": abs(explicit_call["price"] - bsm_call),
        "stability_indicator": explicit_call["stability_indicator"]
    },
    {
        "scheme": "implicit",
        "price": implicit_call["price"],
        "bsm_reference": bsm_call,
        "error": implicit_call["price"] - bsm_call,
        "abs_error": abs(implicit_call["price"] - bsm_call),
        "stability_indicator": np.nan
    },
    {
        "scheme": "crank_nicolson",
        "price": crank_call["price"],
        "bsm_reference": bsm_call,
        "error": crank_call["price"] - bsm_call,
        "abs_error": abs(crank_call["price"] - bsm_call),
        "stability_indicator": np.nan
    }
])

call_comparison

## 6. European put validation

The same machinery works for puts.

In [ ]:
put_cfg = FDConfig(option_type="put", m=400, n=1600)

implicit_put = theta_fd(put_cfg, theta=1.0)
crank_put = theta_fd(put_cfg, theta=0.5)

bsm_put = float(bsm_price(
    put_cfg.s0, put_cfg.strike, put_cfg.maturity,
    put_cfg.r, put_cfg.q, put_cfg.sigma, "put"
))

put_comparison = pd.DataFrame([
    {
        "scheme": "implicit",
        "price": implicit_put["price"],
        "bsm_reference": bsm_put,
        "error": implicit_put["price"] - bsm_put,
        "abs_error": abs(implicit_put["price"] - bsm_put)
    },
    {
        "scheme": "crank_nicolson",
        "price": crank_put["price"],
        "bsm_reference": bsm_put,
        "error": crank_put["price"] - bsm_put,
        "abs_error": abs(crank_put["price"] - bsm_put)
    }
])

put_comparison

## 7. Value curve and snapshots

The PDE solver returns the full curve:

$$
S\mapsto V(0,S)
$$
not just the price at $S_0$.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(crank_call["S"], crank_call["V"], label="Crank-Nicolson call")
plt.plot(
    crank_call["S"],
    bsm_price(np.maximum(crank_call["S"], 1e-8), call_cfg.strike, call_cfg.maturity, call_cfg.r, call_cfg.q, call_cfg.sigma, "call"),
    linestyle="--",
    label="BSM reference"
)
plt.axvline(call_cfg.s0, linestyle=":", label="S0")
plt.xlim(0, 220)
plt.title("European Call Value Curve")
plt.xlabel("Spot")
plt.ylabel("Option value")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

for tau, values in crank_call["snapshots"].items():
    plt.plot(crank_call["S"], values, label=f"tau={tau}")

plt.xlim(50, 160)
plt.title("Crank-Nicolson Solution Snapshots")
plt.xlabel("Spot")
plt.ylabel("Value")
plt.legend()
plt.show()

## 8. Finite-difference Greeks

Grid Delta and Gamma are:

$$
\Delta_i \approx \frac{V_{i+1}-V_{i-1}}{2\Delta S}
$$
$$
\Gamma_i \approx \frac{V_{i+1}-2V_i+V_{i-1}}{\Delta S^2}
$$

In [ ]:
def fd_greeks(S, V):
    dS = S[1] - S[0]
    delta = np.full_like(V, np.nan, dtype=float)
    gamma = np.full_like(V, np.nan, dtype=float)
    delta[1:-1] = (V[2:] - V[:-2]) / (2 * dS)
    gamma[1:-1] = (V[2:] - 2 * V[1:-1] + V[:-2]) / dS**2
    return pd.DataFrame({"spot": S, "value": V, "delta": delta, "gamma": gamma})


greeks = fd_greeks(crank_call["S"], crank_call["V"])

fd_delta_s0 = float(np.interp(call_cfg.s0, greeks["spot"], greeks["delta"]))
fd_gamma_s0 = float(np.interp(call_cfg.s0, greeks["spot"], greeks["gamma"]))

bsm_delta_s0 = bsm_delta(call_cfg.s0, call_cfg.strike, call_cfg.maturity, call_cfg.r, call_cfg.q, call_cfg.sigma, "call")
bsm_gamma_s0 = bsm_gamma(call_cfg.s0, call_cfg.strike, call_cfg.maturity, call_cfg.r, call_cfg.q, call_cfg.sigma)

pd.Series({
    "fd_delta_at_s0": fd_delta_s0,
    "bsm_delta_at_s0": bsm_delta_s0,
    "delta_error": fd_delta_s0 - bsm_delta_s0,
    "fd_gamma_at_s0": fd_gamma_s0,
    "bsm_gamma_at_s0": bsm_gamma_s0,
    "gamma_error": fd_gamma_s0 - bsm_gamma_s0
})

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(greeks["spot"], greeks["delta"], label="FD Delta")
plt.axvline(call_cfg.strike, linestyle="--", label="Strike")
plt.xlim(50, 160)
plt.ylim(-0.05, 1.05)
plt.title("Finite Difference Delta")
plt.xlabel("Spot")
plt.ylabel("Delta")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(greeks["spot"], greeks["gamma"], label="FD Gamma")
plt.axvline(call_cfg.strike, linestyle="--", label="Strike")
plt.xlim(50, 160)
plt.title("Finite Difference Gamma")
plt.xlabel("Spot")
plt.ylabel("Gamma")
plt.legend()
plt.show()

## 9. Convergence experiment

A PDE pricer should converge as the grid is refined.

In [ ]:
def convergence_experiment(base_cfg, specs):
    rows = []
    ref = float(bsm_price(
        base_cfg.s0, base_cfg.strike, base_cfg.maturity,
        base_cfg.r, base_cfg.q, base_cfg.sigma, base_cfg.option_type
    ))

    for m, n in specs:
        cfg_i = FDConfig(**{**asdict(base_cfg), "m": m, "n": n})

        for name, theta in {"implicit": 1.0, "crank_nicolson": 0.5}.items():
            result = theta_fd(cfg_i, theta=theta)
            rows.append({
                "scheme": name,
                "m": m,
                "n": n,
                "dS": result["dS"],
                "dt": result["dt"],
                "price": result["price"],
                "reference": ref,
                "error": result["price"] - ref,
                "abs_error": abs(result["price"] - ref)
            })

    return pd.DataFrame(rows)


specs = [(100, 200), (150, 300), (200, 400), (300, 600), (400, 800), (600, 1200)]
convergence = convergence_experiment(FDConfig(option_type="call"), specs)

convergence

In [ ]:
plt.figure(figsize=(10, 6))

for scheme, group in convergence.groupby("scheme"):
    plt.plot(group["m"], group["abs_error"], marker="o", label=scheme)

plt.xscale("log")
plt.yscale("log")
plt.title("Finite Difference Convergence vs BSM")
plt.xlabel("Spot grid steps")
plt.ylabel("Absolute error")
plt.legend()
plt.show()

## 10. American put via projected finite differences

The American put satisfies:

$$
V(\tau,S)\geq(K-S)^+
$$
The PDE becomes a linear complementarity problem. We approximate it using projected successive over-relaxation.

In [ ]:
am_put_cfg = FDConfig(option_type="put", m=400, n=800)

american_put = theta_fd(am_put_cfg, theta=0.5, american=True)
european_put = theta_fd(am_put_cfg, theta=0.5, american=False)

american_summary = pd.Series({
    "american_put_price": american_put["price"],
    "european_put_price": european_put["price"],
    "early_exercise_premium": american_put["price"] - european_put["price"],
    "mean_psor_iterations": np.mean(american_put["psor_iterations"]),
    "max_psor_iterations": np.max(american_put["psor_iterations"])
})

american_summary

## 11. Early-exercise boundary

The exercise boundary is the highest spot where immediate exercise is approximately optimal.

In [ ]:
def exercise_boundary_from_snapshots(S, snapshots, K, tol=1e-4):
    rows = []
    intrinsic = np.maximum(K - S, 0.0)

    for tau, V in snapshots.items():
        mask = (intrinsic > 0) & (np.abs(V - intrinsic) < tol)
        boundary = float(np.max(S[mask])) if np.any(mask) else np.nan
        rows.append({
            "tau": float(tau),
            "exercise_boundary": boundary,
            "n_exercise_nodes": int(mask.sum())
        })

    return pd.DataFrame(rows).sort_values("tau")


exercise_boundary = exercise_boundary_from_snapshots(
    american_put["S"],
    american_put["snapshots"],
    am_put_cfg.strike
)

exercise_boundary

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(exercise_boundary["tau"], exercise_boundary["exercise_boundary"], marker="o")
plt.axhline(am_put_cfg.strike, linestyle="--", label="Strike")
plt.title("Approximate American Put Exercise Boundary")
plt.xlabel("Time to maturity")
plt.ylabel("Exercise boundary spot")
plt.legend()
plt.show()

S = american_put["S"]
intrinsic = np.maximum(am_put_cfg.strike - S, 0.0)

plt.figure(figsize=(10, 6))
plt.plot(S, american_put["V"], label="American put")
plt.plot(S, european_put["V"], linestyle="--", label="European put")
plt.plot(S, intrinsic, linestyle=":", label="Intrinsic")
plt.xlim(40, 160)
plt.title("American vs European Put")
plt.xlabel("Spot")
plt.ylabel("Value")
plt.legend()
plt.show()

## 12. Explicit stability warning

The explicit method is conditionally stable.

A rough practical warning metric is:

$$
\sigma^2M^2\Delta t
$$
Large values are dangerous and can produce oscillations or nonsensical values.

In [ ]:
unstable_cfg = FDConfig(option_type="call", m=400, n=80)
unstable = explicit_fd(unstable_cfg)

stability_demo = pd.Series({
    "explicit_price_coarse_time_grid": unstable["price"],
    "stability_indicator": unstable["stability_indicator"],
    "min_grid_value": float(np.min(unstable["V"])),
    "max_grid_value": float(np.max(unstable["V"]))
})

stability_demo

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(unstable["S"], unstable["V"])
plt.xlim(0, 220)
plt.title("Explicit Scheme with Coarse Time Grid")
plt.xlabel("Spot")
plt.ylabel("Value")
plt.show()

## 13. Grid error diagnostic

For European options, we can compare the whole finite-difference value curve against the analytical BSM curve.

In [ ]:
def grid_error_vs_bsm(result, cfg):
    S = result["S"]
    ref = bsm_price(
        np.maximum(S, 1e-8),
        cfg.strike,
        cfg.maturity,
        cfg.r,
        cfg.q,
        cfg.sigma,
        cfg.option_type
    )
    return pd.DataFrame({
        "spot": S,
        "fd_value": result["V"],
        "bsm_value": ref,
        "error": result["V"] - ref,
        "abs_error": np.abs(result["V"] - ref)
    })


grid_error = grid_error_vs_bsm(crank_call, call_cfg)

grid_error[(grid_error["spot"] >= 40) & (grid_error["spot"] <= 180)]["abs_error"].describe()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(grid_error["spot"], grid_error["error"])
plt.xlim(40, 180)
plt.axhline(0, linestyle="--")
plt.title("Crank-Nicolson Grid Error vs BSM")
plt.xlabel("Spot")
plt.ylabel("FD - BSM")
plt.show()

## 14. Saving outputs

The notebook saves pricing comparisons, convergence results, value curves, Greeks, American exercise boundary, and diagnostics.

In [ ]:
output_dir = Path("data/processed/finite_difference_pde_pricer_v1")
output_dir.mkdir(parents=True, exist_ok=True)

call_comparison_path = output_dir / "european_call_pricing_comparison.csv"
put_comparison_path = output_dir / "european_put_pricing_comparison.csv"
convergence_path = output_dir / "convergence_results.csv"
call_curve_path = output_dir / "crank_nicolson_call_curve.csv"
greeks_path = output_dir / "crank_nicolson_call_greeks.csv"
american_curve_path = output_dir / "american_put_curve.csv"
exercise_boundary_path = output_dir / "american_put_exercise_boundary.csv"
grid_error_path = output_dir / "grid_error_vs_bsm.csv"
manifest_path = output_dir / "manifest.json"

call_comparison.to_csv(call_comparison_path, index=False)
put_comparison.to_csv(put_comparison_path, index=False)
convergence.to_csv(convergence_path, index=False)

pd.DataFrame({"spot": crank_call["S"], "value": crank_call["V"]}).to_csv(call_curve_path, index=False)
greeks.to_csv(greeks_path, index=False)

pd.DataFrame({
    "spot": american_put["S"],
    "american_put": american_put["V"],
    "european_put": european_put["V"],
    "intrinsic": intrinsic
}).to_csv(american_curve_path, index=False)

exercise_boundary.to_csv(exercise_boundary_path, index=False)
grid_error.to_csv(grid_error_path, index=False)

manifest = {
    "dataset_name": "finite_difference_pde_pricer_outputs",
    "schema_version": "finite_difference_pde_pricer_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "call_config": asdict(call_cfg),
    "american_put_config": asdict(am_put_cfg),
    "bsm_call_reference": float(bsm_call),
    "crank_nicolson_call_price": float(crank_call["price"]),
    "crank_nicolson_call_abs_error": float(abs(crank_call["price"] - bsm_call)),
    "american_put_price": float(american_put["price"]),
    "european_put_price": float(european_put["price"]),
    "early_exercise_premium": float(american_put["price"] - european_put["price"]),
    "notes": [
        "PDE solved in time-to-maturity tau.",
        "European options validated against analytical BSM.",
        "Implicit and Crank-Nicolson use Thomas tridiagonal solver.",
        "American put uses projected SOR.",
        "Explicit stability warning included."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

manifest_path

## 15. Practical checklist

Before trusting a finite difference pricer, check:

1. PDE sign convention;
2. terminal payoff;
3. boundary conditions;
4. domain size $S_{\max}$;
5. grid resolution;
6. explicit stability;
7. tridiagonal solver correctness;
8. interpolation at $S_0$;
9. convergence under refinement;
10. Greek stability;
11. American exercise projection;
12. saved diagnostics.

## 16. Limitations

This is an educational but serious implementation.

Limitations:

- uniform spot grid;
- simple boundaries;
- no Rannacher smoothing;
- basic PSOR American solver;
- constant $r,q,\sigma$;
- one underlying only;
- no local volatility, stochastic volatility, or jumps.

Production PDE pricers often use non-uniform grids, damping, higher-order schemes, ADI methods, and stronger validation.

## 17. Research frontier and extensions

Important extensions:

1. **Rannacher smoothing** for payoff kinks;
2. **non-uniform grids** around strikes/barriers;
3. **local volatility PDEs** from Dupire surfaces;
4. **jump-diffusion PIDEs** with integral terms;
5. **ADI schemes** for Heston;
6. **penalty and policy-iteration methods** for American options;
7. **GPU/C++ finite difference kernels**;
8. **deep BSDE and neural PDE solvers** for high-dimensional problems.

## 18. Suggested follow-up notebooks

1. **02_11_heston_model_calibration**
2. **02_14_local_volatility_dupire_surface**
3. **02_15_jump_diffusion_pide_pricer**
4. **02_13_multilevel_monte_carlo_pricing**
5. **06_10_numba_cpp_pde_kernels**
6. **07_02_volatility_surface_pricing_and_alpha**

## 19. Summary

This notebook implemented a finite difference PDE pricer.

It showed how to:

1. transform the BSM PDE into time-to-maturity form;
2. define terminal and boundary conditions;
3. implement explicit, implicit, and Crank-Nicolson schemes;
4. solve tridiagonal systems;
5. validate European prices against BSM;
6. compute finite-difference Delta and Gamma;
7. run convergence tests;
8. demonstrate explicit instability risk;
9. price an American put with projected SOR;
10. extract an early-exercise boundary.

The key computational takeaway:

> A PDE pricer is a numerical engine. Stability, boundary conditions, interpolation, and convergence matter as much as the financial formula.

The key financial takeaway:

> Finite differences extend replication theory into a flexible framework for early exercise and more complex model dynamics.

## 20. Further reading

- Wilmott, *Paul Wilmott on Quantitative Finance*.
- Tavella and Randall, *Pricing Financial Instruments: The Finite Difference Method*.
- Duffy, *Finite Difference Methods in Financial Engineering*.
- Shreve, *Stochastic Calculus for Finance II*.
- Björk, *Arbitrage Theory in Continuous Time*.

Extensions:

- Rannacher smoothing;
- non-uniform grids;
- local volatility PDEs;
- Heston ADI schemes;
- jump-diffusion PIDEs;
- neural PDE solvers.